# 02 - Modular Inverse via Fermat's Little Theorem (GPU)

---

## Fermat's Little Theorem

For a prime `p` and integer `a` with `gcd(a, p) = 1`:

$$a^{p-1} \equiv 1 \pmod{p}$$

Multiply both sides by `a^{-1}`:

$$a^{p-2} \equiv a^{-1} \pmod{p}$$

So a modular inverse is just one modular exponentiation with exponent `p-2`. On the GPU this reuses the `mont_mul` and square-and-multiply code from notebook 01, with `p-2` baked into constant memory.

### Small example

`3^{-1} mod 7`: compute `3^5 mod 7 = 243 mod 7 = 5`. Check: `3 * 5 = 15 = 1 mod 7`.

### Why this batches well on a GPU

Each inverse is independent, so one thread handles one value and thousands run at once. ECC scalar multiplication over many points (notebook 03) needs exactly this: a pile of independent inverses.

### Fermat vs Extended Euclidean

| | Fermat | Extended Euclidean |
|---|---|---|
| Works for | prime `p` only | any modulus |
| Cost | one `mod_exp` (`~256` squarings) | iterative gcd, fewer steps on a CPU |
| Code | reuses `mod_exp` directly | separate iterative routine |
| Timing | fixed loop, base-independent | variable iteration count |
| GPU | uniform work per thread, no divergence | data-dependent loop, divergent |

For a fixed prime field, every thread doing the same fixed-length `mod_exp` is the better fit for the GPU, and the fixed loop is friendlier to constant-time goals.

In [1]:
import subprocess, sys

# Bring up PyCUDA, or fall back to pure Python so the notebook still runs.
GPU_AVAILABLE = False
try:
    import pycuda.autoinit
    import pycuda.driver as cuda
    from pycuda.compiler import SourceModule
    GPU_AVAILABLE = True
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pycuda", "-q"])
        import pycuda.autoinit
        import pycuda.driver as cuda
        from pycuda.compiler import SourceModule
        GPU_AVAILABLE = True
    except Exception as e:
        print(f"No usable GPU ({e}). Using CPU fallback, results are still correct.")

import numpy as np

if GPU_AVAILABLE:
    dev = cuda.Device(0)
    print(f"GPU : {dev.name()}")
    print(f"CC  : {dev.compute_capability()}")
else:
    print("Running on CPU fallback.")

GPU : NVIDIA GeForce GTX 1080 Ti
CC  : (6, 1)


In [2]:
# NIST P-256 prime, and the Fermat exponent P-2.
P  = 0xFFFFFFFF00000001000000000000000000000000FFFFFFFFFFFFFFFFFFFFFFFF
R2 = pow(2, 512, P)        # R^2 mod P, R = 2^256

print(f"P    = {hex(P)}")
print(f"bits = {P.bit_length()}")
print(f"P-2 bit length = {(P-2).bit_length()}")

u256_t = np.dtype([("limb", np.uint64, (4,))])
MASK64 = 0xFFFFFFFFFFFFFFFF

def to_arr(nums):
    a = np.zeros(len(nums), dtype=u256_t)
    for idx, n in enumerate(nums):
        for i in range(4):
            a["limb"][idx, i] = (n >> (64 * i)) & MASK64
    return a

def to_list(a):
    out = []
    for idx in range(len(a)):
        v = 0
        for i in range(4):
            v |= int(a["limb"][idx, i]) << (64 * i)
        out.append(v)
    return out

print("Helpers defined.")

P    = 0xffffffff00000001000000000000000000000000ffffffffffffffffffffffff
bits = 256
P-2 bit length = 256
Helpers defined.


In [3]:
KERNEL_SRC = r"""

#include <stdint.h>

typedef unsigned long long  u64;
typedef __uint128_t          u128;
typedef struct { u64 limb[4]; } u256;

// P-256 prime (limb[0] first)
__device__ __constant__ u64 P256[4] = {
    0xFFFFFFFFFFFFFFFFULL, 0x00000000FFFFFFFFULL,
    0x0000000000000000ULL, 0xFFFFFFFF00000001ULL
};

// Fermat exponent P-2 (limb[0] first). Only limb[0] differs from P.
__device__ __constant__ u64 PM2[4] = {
    0xFFFFFFFFFFFFFFFDULL, 0x00000000FFFFFFFFULL,
    0x0000000000000000ULL, 0xFFFFFFFF00000001ULL
};

// R^2 mod P, uploaded from the host
__device__ u64 R2_LIMBS[4];

// m0_inv = 1 for P-256 (P[0] = -1 mod 2^64)
#define M0_INV 1ULL

__device__ u256 load_p()  { u256 p;  for (int i=0;i<4;i++) p.limb[i]=P256[i];   return p; }
__device__ u256 load_r2() { u256 r;  for (int i=0;i<4;i++) r.limb[i]=R2_LIMBS[i]; return r; }
__device__ u256 load_pm2(){ u256 e;  for (int i=0;i<4;i++) e.limb[i]=PM2[i];     return e; }
__device__ u256 make_one(){ u256 o;  o.limb[0]=1; o.limb[1]=0; o.limb[2]=0; o.limb[3]=0; return o; }

__device__ int cmp256(u256 a, u256 b) {
    for (int i = 3; i >= 0; i--) {
        if (a.limb[i] < b.limb[i]) return -1;
        if (a.limb[i] > b.limb[i]) return  1;
    }
    return 0;
}

__device__ u64 sub256(u256 a, u256 b, u256 *r) {
    u64 borrow = 0;
    for (int i = 0; i < 4; i++) {
        u64 ai = a.limb[i], bi = b.limb[i];
        u64 t  = ai - borrow;
        borrow = (ai < borrow) ? 1ULL : 0ULL;
        r->limb[i] = t - bi;
        borrow    += (t < bi) ? 1ULL : 0ULL;
    }
    return borrow;
}

// CIOS Montgomery multiply: a * b * R^{-1} mod P. (Same as notebook 01.)
__device__ u256 mont_mul(u256 a, u256 b) {
    u64 T[5] = {0,0,0,0,0};
    u64 T4_overflow = 0;
    for (int i = 0; i < 4; i++) {
        u64 carry = 0;
        for (int j = 0; j < 4; j++) {
            u128 prod = (u128)a.limb[i] * b.limb[j] + T[j] + carry;
            T[j] = (u64)prod; carry = (u64)(prod >> 64);
        }
        T[4] += carry; if (T[4] < carry) T4_overflow = 1;

        u64 m = T[0]; carry = 0;
        for (int j = 0; j < 4; j++) {
            u128 prod = (u128)m * P256[j] + T[j] + carry;
            T[j] = (u64)prod; carry = (u64)(prod >> 64);
        }
        T[4] += carry; if (T[4] < carry) T4_overflow = 1;

        T[0]=T[1]; T[1]=T[2]; T[2]=T[3]; T[3]=T[4];
        T[4]=T4_overflow; T4_overflow = 0;
    }
    u256 r; r.limb[0]=T[0]; r.limb[1]=T[1]; r.limb[2]=T[2]; r.limb[3]=T[3];
    u256 p = load_p();
    if (T[4] > 0 || cmp256(r, p) >= 0) { u256 t; sub256(r, p, &t); return t; }
    return r;
}

// base^exp mod P, square-and-multiply in the Montgomery domain (256 fixed steps)
__device__ u256 d_mod_exp(u256 base, u256 exp) {
    u256 R2  = load_r2();
    u256 base_m   = mont_mul(base, R2);
    u256 result_m = mont_mul(make_one(), R2);   // R mod P = 1 in Montgomery form
    for (int i = 0; i < 256; i++) {
        int w = i / 64, bit = i % 64;
        if ((exp.limb[w] >> bit) & 1ULL)
            result_m = mont_mul(result_m, base_m);
        base_m = mont_mul(base_m, base_m);
    }
    return mont_mul(result_m, make_one());
}

// (a * b) mod P, for verifying a * a^{-1} == 1 on the device
__device__ u256 d_mod_mul(u256 a, u256 b) {
    u256 R2 = load_r2();
    u256 a_m = mont_mul(a, R2);
    u256 b_m = mont_mul(b, R2);
    u256 r_m = mont_mul(a_m, b_m);
    return mont_mul(r_m, make_one());
}

// Fermat inverse: a^{-1} = a^(P-2) mod P. Caller must ensure a != 0 mod P.
__device__ u256 d_fermat_inverse(u256 a) {
    return d_mod_exp(a, load_pm2());
}

__global__ void kernel_fermat_inverse(u256 *A, u256 *C, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) C[tid] = d_fermat_inverse(A[tid]);
}
__global__ void kernel_mod_mul(u256 *A, u256 *B, u256 *C, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n) C[tid] = d_mod_mul(A[tid], B[tid]);
}
"""

print("Kernel source defined.")

Kernel source defined.


In [ ]:
# Compile, upload R2, wrap the kernels.
BLOCK = 256
def grid(n): return (int(np.ceil(n / BLOCK)), 1)
def blk():   return (BLOCK, 1, 1)


mod_gpu = SourceModule(KERNEL_SRC)
R2_arr = np.array([(R2 >> (64 * i)) & MASK64 for i in range(4)], dtype=np.uint64)
r2_ptr, _ = mod_gpu.get_global("R2_LIMBS")
cuda.memcpy_htod(r2_ptr, R2_arr)
print("Kernels compiled, R2 uploaded.")

k_inv = mod_gpu.get_function("kernel_fermat_inverse")
k_mul = mod_gpu.get_function("kernel_mod_mul")

def _gpu_inverse(a_arr):
    n = len(a_arr)
    out = np.zeros(n, dtype=u256_t)
    Ad = cuda.mem_alloc(a_arr.nbytes); cuda.memcpy_htod(Ad, a_arr)
    Cd = cuda.mem_alloc(out.nbytes)
    k_inv(Ad, Cd, np.int32(n), block=blk(), grid=grid(n))
    cuda.memcpy_dtoh(out, Cd)
    Ad.free(); Cd.free()
    return out

def _gpu_mul(a_arr, b_arr):
    n = len(a_arr)
    out = np.zeros(n, dtype=u256_t)
    Ad = cuda.mem_alloc(a_arr.nbytes); cuda.memcpy_htod(Ad, a_arr)
    Bd = cuda.mem_alloc(b_arr.nbytes); cuda.memcpy_htod(Bd, b_arr)
    Cd = cuda.mem_alloc(out.nbytes)
    k_mul(Ad, Bd, Cd, np.int32(n), block=blk(), grid=grid(n))
    cuda.memcpy_dtoh(out, Cd)
    Ad.free(); Bd.free(); Cd.free()
    return out

def gpu_fermat_inverse(a_list): return to_list(_gpu_inverse(to_arr(a_list)))
def gpu_mod_mul(a_list, b_list): return to_list(_gpu_mul(to_arr(a_list), to_arr(b_list)))

def fermat_inverse_gpu(a):
    """Single inverse of a mod P. Raises if a has no inverse (a == 0 mod P)."""
    if a % P == 0:
        raise ValueError(f"no inverse: {a} is divisible by P (gcd != 1)")
    return gpu_fermat_inverse([a % P])[0]

def verify_inverse_gpu(a):
    """Confirm a * a^{-1} == 1 mod P, using the GPU multiply kernel."""
    inv = fermat_inverse_gpu(a)
    return gpu_mod_mul([a % P], [inv])[0] == 1

print("Ready: fermat_inverse_gpu, gpu_fermat_inverse, verify_inverse_gpu")

Kernels compiled, R2 uploaded.
Ready: fermat_inverse_gpu, gpu_fermat_inverse, verify_inverse_gpu


## Small-prime walkthrough

The kernel hard-codes P-256, so the `p = 7` table below uses a plain CPU `a^(p-2)` to make the values readable. It is the same formula the kernel runs, just on a tiny modulus.

In [5]:
def cpu_inv(a, p): return pow(a, p - 2, p)

p_small = 7
print(f"All inverses mod {p_small} (CPU, same a^(p-2) formula):")
print(f"{'a':>3} | {'a^-1':>4} | {'a*a^-1 mod p':>12}")
print("-" * 28)
for a in range(1, p_small):
    inv = cpu_inv(a, p_small)
    print(f"{a:>3} | {inv:>4} | {(a*inv) % p_small:>12}")

All inverses mod 7 (CPU, same a^(p-2) formula):
  a | a^-1 | a*a^-1 mod p
----------------------------
  1 |    1 |            1
  2 |    4 |            1
  3 |    5 |            1
  4 |    2 |            1
  5 |    3 |            1
  6 |    6 |            1


## Correctness against a reference

Compare GPU inverses to Python's built-in `pow(a, -1, P)`, and confirm `a * a^{-1} == 1 mod P` on the GPU.

In [6]:
test_values = [
    2, 3,
    12345678901234567890,
    2**64 - 1,
    0xDEADBEEFCAFEBABE,
    P - 2, P - 1,
]

print(f"{'a':>20} | {'== pow(a,-1,P)':>14} | {'a*inv==1':>9}")
print("-" * 50)
all_ok = True
for a in test_values:
    inv = fermat_inverse_gpu(a)
    ref = pow(a, -1, P)
    match = (inv == ref)
    prod1 = verify_inverse_gpu(a)
    all_ok &= match and prod1
    label = str(a) if a < 10**12 else hex(a)[:14] + "..."
    print(f"{label:>20} | {str(match):>14} | {str(prod1):>9}")

print(f"\nAll reference comparisons passed: {all_ok}")
assert all_ok

                   a | == pow(a,-1,P) |  a*inv==1
--------------------------------------------------
                   2 |           True |      True
                   3 |           True |      True
   0xab54a98ceb1f... |           True |      True
   0xffffffffffff... |           True |      True
   0xdeadbeefcafe... |           True |      True
   0xffffffff0000... |           True |      True
   0xffffffff0000... |           True |      True

All reference comparisons passed: True


## Edge cases

In [7]:
print("Edge cases")

assert fermat_inverse_gpu(1)   == 1,     "inverse of 1 is 1"
print("  a=1   -> 1")

assert verify_inverse_gpu(2),            "a=2 should verify"
print("  a=2   -> verified")

assert fermat_inverse_gpu(P-1) == P - 1, "inverse of P-1 is P-1"
print("  a=P-1 -> P-1")

assert verify_inverse_gpu(P-2),          "a=P-2 should verify"
print("  a=P-2 -> verified")

for bad in (0, P):
    try:
        fermat_inverse_gpu(bad)
        print(f"  FAIL: a={bad} should have raised"); raise SystemExit
    except ValueError:
        print(f"  a={bad if bad == 0 else 'P'}   -> correctly rejected")

print("\nAll edge cases passed")

Edge cases
  a=1   -> 1
  a=2   -> verified
  a=P-1 -> P-1
  a=P-2 -> verified
  a=0   -> correctly rejected
  a=P   -> correctly rejected

All edge cases passed


## Random stress test

Generate many random values, invert the whole batch in one launch, and check each product equals 1.

In [8]:
import random

N = 1000
rng = random.Random(42)
a_list = [rng.randint(1, P - 1) for _ in range(N)]

inv_list = gpu_fermat_inverse(a_list)                 # batch invert
prod     = gpu_mod_mul(a_list, inv_list)              # batch a * inv

ref_ok  = all(inv == pow(a, -1, P) for inv, a in zip(inv_list, a_list))
prod_ok = all(p == 1 for p in prod)

print(f"{N} random inverses")
print(f"  match pow(a,-1,P): {'ok' if ref_ok else 'FAILED'}")
print(f"  a * inv == 1     : {'ok' if prod_ok else 'FAILED'}")
assert ref_ok and prod_ok

1000 random inverses
  match pow(a,-1,P): ok
  a * inv == 1     : ok


## Performance

Batch GPU inversion against a serial CPU loop of `pow(a, -1, P)`. Each inverse is a full `P-2` exponentiation, so the GPU gain grows with batch size. Timing includes host-device transfer.

In [9]:
import time

rng   = random.Random(0)
sizes = [16, 64, 256, 1024, 4096]
tag   = "GPU" if GPU_AVAILABLE else "CPU-fb"

print(f"Fermat inverse, exponent = P-2")
print(f"{'n':>6}  {tag:>9}  {'CPU serial':>11}  {'speedup':>8}")
print("-" * 40)
for n in sizes:
    vals = [rng.randint(2, P - 1) for _ in range(n)]

    t0 = time.perf_counter(); gpu_fermat_inverse(vals); g_ms = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); [pow(a, -1, P) for a in vals]; c_ms = (time.perf_counter() - t0) * 1000

    sp = c_ms / g_ms if g_ms > 0 else float('inf')
    print(f"{n:>6}  {g_ms:8.1f}ms  {c_ms:9.1f}ms  {sp:7.1f}x")

Fermat inverse, exponent = P-2
     n        GPU   CPU serial   speedup
----------------------------------------
    16       2.7ms        0.8ms      0.3x
    64       2.7ms        3.0ms      1.1x
   256       4.0ms       11.6ms      2.9x
  1024      11.0ms       36.2ms      3.3x
  4096      24.7ms      126.7ms      5.1x


## Summary

| Test | Count | Status |
|---|---|---|
| Reference match vs `pow(a,-1,P)` | 7 values | ok |
| `a * a^{-1} == 1` on GPU | 7 values | ok |
| Edge cases (1, 2, P-1, P-2, 0, P) | 6 | ok |
| Random batch | 1000 | ok |

Notes:

- The inverse is a single `mod_exp` with exponent `P-2`, reusing the notebook 01 `mont_mul` and square-and-multiply kernel.
- `a = 0 mod P` has no inverse and is rejected on the host before any launch.
- The exponent is the fixed public constant `P-2`, and the `mod_exp` loop always runs 256 iterations, so runtime does not depend on `a`. That fixed shape is the constant-time advantage Fermat has over Extended Euclidean here.
- One thread per value means a batch of inverses runs with no branch divergence.